In [1]:
%load_ext autoreload
%autoreload 2

## Exp1 Postprocessing Results

In [40]:
import pandas as pd
import plotly.express as px

In [5]:
df = pd.read_csv("./execution_time_comparison/exp1/root_screening_lingam.csv")

In [6]:
import pandas as pd
import plotly.express as px

# --- build tidy df with metrics for both methods ---

# Roots screening
roots_df = df[
    ['precision_roots_screening',
     'recall_roots_screening',
     'mse_roots_screening',
     'shd_roots_screening',
     'p', 's', 'n', 'root_frac']
].copy()

roots_df = roots_df.rename(columns={
    'precision_roots_screening': 'precision',
    'recall_roots_screening': 'recall',
    'mse_roots_screening': 'mse',
    'shd_roots_screening': 'shd'
})
roots_df['method'] = 'Roots screening'

# DirectLiNGAM
dl_df = df[
    ['precision_direct_lingam',
     'recall_direct_lingam',
     'mse_direct_lingam',
     'shd_direct_lingam',
     'p', 's', 'n', 'root_frac']
].copy()

dl_df = dl_df.rename(columns={
    'precision_direct_lingam': 'precision',
    'recall_direct_lingam': 'recall',
    'mse_direct_lingam': 'mse',
    'shd_direct_lingam': 'shd'
})
dl_df['method'] = 'DirectLiNGAM'

# stack
metrics_df = pd.concat([roots_df, dl_df], ignore_index=True)

# melt to long format: one row per (run, metric, method)
metrics_long = metrics_df.melt(
    id_vars=['method', 'p', 's', 'n', 'root_frac'],
    value_vars=['precision', 'recall', 'mse', 'shd'],
    var_name='metric',
    value_name='value'
)

# enforce category order (optional but helps)
metrics_long['metric'] = pd.Categorical(
    metrics_long['metric'],
    categories=['precision', 'recall', 'mse', 'shd'],
    ordered=True
)
metrics_long['root_frac'] = metrics_long['root_frac'].astype(float)


In [15]:
metrics_long[metrics_long['value']<4000000]

,method,p,s,n,root_frac,metric,value
0,Roots screening,100,0.15,10000,0.1,precision,0.993207
1,Roots screening,100,0.15,10000,0.1,precision,0.992084
2,Roots screening,100,0.15,10000,0.1,precision,0.995995
3,Roots screening,100,0.15,10000,0.1,precision,0.989362
4,Roots screening,100,0.15,10000,0.1,precision,0.985825
...,...,...,...,...,...,...,...
1435,DirectLiNGAM,500,0.15,10000,0.3,shd,11695.000000
1436,DirectLiNGAM,500,0.15,10000,0.3,shd,11778.000000
1437,DirectLiNGAM,500,0.15,10000,0.3,shd,11673.000000
1438,DirectLiNGAM,500,0.15,10000,0.3,shd,12496.000000


In [16]:
metrics_long = metrics_long[metrics_long['value']<4000000]

In [31]:
fig_metrics = px.box(
    metrics_long[(metrics_long['metric'] != "shd") & (metrics_long['metric'] != "mse")],
    x='root_frac',
    y='value',
    color='method',
    facet_row='metric',
    facet_col='p',
    category_orders={
        'metric': ['precision', 'recall'],
        'root_frac': [0.1, 0.2, 0.3],
        'p': [100, 300, 500]
    },
    points='all',  # show all runs
)

fig_metrics.update_layout(
    title='Roots screening vs DirectLiNGAM across Precision, Recall and root_frac',
    boxmode='group',
    yaxis_title='Metric value'
)
fig_metrics.show()


In [32]:
fig_metrics.write_html("./execution_time_comparison/exp1/charts/precision_recall.html")

In [33]:
fig_metrics = px.box(
    metrics_long[(metrics_long['metric'] == "shd")],
    x='root_frac',
    y='value',
    color='method',
    facet_row='metric',
    facet_col='p',
    category_orders={
        'metric': ['shd'],
        'root_frac': [0.1, 0.2, 0.3],
        'p': [100, 300, 500]
    },
    points='all',  # show all runs
)

fig_metrics.update_layout(
    title='Roots screening vs DirectLiNGAM SHD vs root_frac',
    boxmode='group',
    yaxis_title='Metric value'
)
fig_metrics.show()


In [34]:
fig_metrics.write_html("./execution_time_comparison/exp1/charts/shd.html")

In [35]:
fig_metrics = px.box(
    metrics_long[(metrics_long['metric'] == "mse")],
    x='root_frac',
    y='value',
    color='method',
    facet_row='metric',
    facet_col='p',
    category_orders={
        'metric': ['mse'],
        'root_frac': [0.1, 0.2, 0.3],
        'p': [100, 300, 500]
    },
    points='all',  # show all runs
)

fig_metrics.update_layout(
    title='Roots screening vs DirectLiNGAM MSE vs root_frac',
    boxmode='group',
    yaxis_title='Metric value'
)
fig_metrics.show()


In [36]:
fig_metrics.write_html("./execution_time_comparison/exp1/charts/mse.html")

## runtime 

In [23]:
# tidy execution time
time_long = df.melt(
    id_vars=['p', 's', 'n', 'root_frac', 'random_state'],
    value_vars=['exec_time_sec_roots_screening', 'exec_time_sec_direct_lingam'],
    var_name='method_raw',
    value_name='exec_time_sec'
)

time_long['method'] = time_long['method_raw'].map({
    'exec_time_sec_roots_screening': 'Roots screening',
    'exec_time_sec_direct_lingam': 'DirectLiNGAM'
})

time_long['root_frac'] = time_long['root_frac'].astype(float)

In [24]:
fig_time = px.box(
    time_long,
    x='root_frac',
    y='exec_time_sec',
    color='method',
    facet_col='p',
    category_orders={
        'root_frac': [0.1, 0.2, 0.3],
        'p': [100, 300, 500]
    },
    points='all'
)

fig_time.update_layout(
    title='Execution time vs root_frac',
    yaxis_title='Execution time (seconds)',
    boxmode='group'
)
fig_time.show()

In [30]:
fig_time.write_html("roots_screening_vs_direct_lingam_execution_time.html")

In [29]:
time_long[time_long['p']==500]

,p,s,n,root_frac,random_state,method_raw,exec_time_sec,method
120,500,0.15,10000,0.1,0,exec_time_sec_roots_screening,92523.977915,Roots screening
121,500,0.15,10000,0.1,1,exec_time_sec_roots_screening,96660.922576,Roots screening
122,500,0.15,10000,0.1,2,exec_time_sec_roots_screening,94005.908728,Roots screening
123,500,0.15,10000,0.1,3,exec_time_sec_roots_screening,90842.992641,Roots screening
124,500,0.15,10000,0.1,4,exec_time_sec_roots_screening,113298.775791,Roots screening
...,...,...,...,...,...,...,...,...
355,500,0.15,10000,0.3,15,exec_time_sec_direct_lingam,111110.983561,DirectLiNGAM
356,500,0.15,10000,0.3,16,exec_time_sec_direct_lingam,89281.577300,DirectLiNGAM
357,500,0.15,10000,0.3,17,exec_time_sec_direct_lingam,119724.031128,DirectLiNGAM
358,500,0.15,10000,0.3,18,exec_time_sec_direct_lingam,103546.592547,DirectLiNGAM


### Experiment 2

In [3]:
df = pd.read_csv("./execution_time_comparison/exp2/root_screening_lingam_ba_network.csv")

In [4]:
df.head()

,precision_roots_screening,recall_roots_screening,mse_roots_screening,shd_roots_screening,precision_direct_lingam,recall_direct_lingam,mse_direct_lingam,shd_direct_lingam,p,m,n,random_state,exec_time_sec_roots_screening,exec_time_sec_direct_lingam
0,0.970588,1.0,0.000006,3,0.933962,1.0,0.000008,7,100,1,2000,0,214.136740,404.671272
1,1.000000,1.0,0.000004,0,0.970588,1.0,0.000005,3,100,1,2000,1,214.501023,347.798625
2,0.942857,1.0,0.000008,6,0.925234,1.0,0.000009,8,100,1,2000,2,220.560543,353.318810
3,0.990000,1.0,0.000004,1,0.908257,1.0,0.000009,10,100,1,2000,3,211.399196,347.895485
4,0.970588,1.0,0.000005,3,0.961165,1.0,0.000004,4,100,1,2000,4,208.034619,353.876108


In [7]:
# --- build tidy df with metrics for both methods ---

# Roots screening
roots_df = df[
    ['precision_roots_screening',
     'recall_roots_screening',
     'mse_roots_screening',
     'shd_roots_screening',
     'p', 'n', 'm']
].copy()

roots_df = roots_df.rename(columns={
    'precision_roots_screening': 'precision',
    'recall_roots_screening': 'recall',
    'mse_roots_screening': 'mse',
    'shd_roots_screening': 'shd'
})
roots_df['method'] = 'Roots screening'

# DirectLiNGAM
dl_df = df[
    ['precision_direct_lingam',
     'recall_direct_lingam',
     'mse_direct_lingam',
     'shd_direct_lingam',
     'p', 'n', 'm']
].copy()

dl_df = dl_df.rename(columns={
    'precision_direct_lingam': 'precision',
    'recall_direct_lingam': 'recall',
    'mse_direct_lingam': 'mse',
    'shd_direct_lingam': 'shd'
})
dl_df['method'] = 'DirectLiNGAM'

# stack
metrics_df = pd.concat([roots_df, dl_df], ignore_index=True)

# melt to long format: one row per (run, metric, method)
metrics_long = metrics_df.melt(
    id_vars=['method', 'p', 'n', 'm'],
    value_vars=['precision', 'recall', 'mse', 'shd'],
    var_name='metric',
    value_name='value'
)

# enforce category order (optional but helps)
metrics_long['metric'] = pd.Categorical(
    metrics_long['metric'],
    categories=['precision', 'recall', 'mse', 'shd'],
    ordered=True
)
metrics_long['m'] = metrics_long['m'].astype(float)


In [15]:
metrics_long[(metrics_long['p'] == 100) & (metrics_long['n']==2000)]

,method,p,n,m,metric,value
0,Roots screening,100,2000,1.0,precision,0.970588
1,Roots screening,100,2000,1.0,precision,1.000000
2,Roots screening,100,2000,1.0,precision,0.942857
3,Roots screening,100,2000,1.0,precision,0.990000
4,Roots screening,100,2000,1.0,precision,0.970588
...,...,...,...,...,...,...
635,DirectLiNGAM,100,2000,1.0,shd,4.000000
636,DirectLiNGAM,100,2000,1.0,shd,7.000000
637,DirectLiNGAM,100,2000,1.0,shd,5.000000
638,DirectLiNGAM,100,2000,1.0,shd,3.000000


In [26]:
N_LIST = [2000, 6000, 10000]

In [28]:
for n in N_LIST:
    fig_metrics = px.box(
        metrics_long[(metrics_long['metric'] != "shd") & (metrics_long['metric'] != "mse") & (metrics_long['n']==n)],
        x='m',
        y='value',
        color='method',
        facet_row='metric',
        facet_col='p',
        category_orders={
            'metric': ['precision', 'recall'],
            'm': [1],
            'p': [100, 300, 500]
        },
        points='all',  # show all runs
    )

    fig_metrics.update_layout(
        title='Roots screening vs DirectLiNGAM across Precision, Recall and M',
        boxmode='group',
        yaxis_title='value'
    )
    #fig_metrics.show()
    fig_metrics.write_html(f"roots_screening_vs_direct_lingam_precision_recall_m_{n}_samples.html")


In [29]:
for n in N_LIST:
    fig_metrics = px.box(
        metrics_long[(metrics_long['metric'] == "mse") & (metrics_long['n']==n)],
        x='m',
        y='value',
        color='method',
        facet_row='metric',
        facet_col='p',
        category_orders={
            'metric': ['mse'],
            'm': [1],
            'p': [100, 300, 500]
        },
        points='all',  # show all runs
    )

    fig_metrics.update_layout(
        title=f'Roots screening vs DirectLiNGAM MSE with {n} samples',
        boxmode='group',
        yaxis_title='value'
    )
    #fig_metrics.show()
    fig_metrics.write_html(f"roots_screening_vs_direct_lingam_mse_m_{n}_samples.html")


In [37]:
for n in N_LIST:
    fig_metrics = px.box(
        metrics_long[(metrics_long['metric'] == "shd") & (metrics_long['n']==n)],
        x='m',
        y='value',
        color='method',
        facet_row='metric',
        facet_col='p',
        category_orders={
            'metric': ['shd'],
            'm': [1],
            'p': [100, 300, 500]
        },
        points='all',  # show all runs
    )

    fig_metrics.update_layout(
        title='Roots screening vs DirectLiNGAM SHD and M',
        boxmode='group',
        yaxis_title='value'
    )
    fig_metrics.show()
    fig_metrics.write_html(f"roots_screening_vs_direct_lingam_shd_{n}_samples.html")


## execution time

In [30]:
# tidy execution time
time_long = df.melt(
    id_vars=['p', 'm', 'n', 'random_state'],
    value_vars=['exec_time_sec_roots_screening', 'exec_time_sec_direct_lingam'],
    var_name='method_raw',
    value_name='exec_time_sec'
)

time_long['method'] = time_long['method_raw'].map({
    'exec_time_sec_roots_screening': 'Roots screening',
    'exec_time_sec_direct_lingam': 'DirectLiNGAM'
})

time_long['m'] = time_long['m'].astype(float)

In [39]:
time_long[(time_long['p'] == 500) & (time_long['method']=="Roots screening") & (time_long['n'] == 10_000)]

,p,m,n,random_state,method_raw,exec_time_sec,method
80,500,1.0,10000,0,exec_time_sec_roots_screening,50205.635281,Roots screening
81,500,1.0,10000,1,exec_time_sec_roots_screening,50020.900890,Roots screening
82,500,1.0,10000,2,exec_time_sec_roots_screening,51014.838444,Roots screening
83,500,1.0,10000,3,exec_time_sec_roots_screening,48337.663724,Roots screening
84,500,1.0,10000,4,exec_time_sec_roots_screening,49484.971345,Roots screening
85,500,1.0,10000,5,exec_time_sec_roots_screening,53123.485423,Roots screening
86,500,1.0,10000,6,exec_time_sec_roots_screening,48440.158111,Roots screening
87,500,1.0,10000,7,exec_time_sec_roots_screening,47654.371913,Roots screening
88,500,1.0,10000,8,exec_time_sec_roots_screening,49125.982190,Roots screening
89,500,1.0,10000,9,exec_time_sec_roots_screening,44244.270839,Roots screening


In [34]:
for n in N_LIST:
    fig_time = px.box(
        time_long[time_long['n']==n],
        x='m',
        y='exec_time_sec',
        color='method',
        facet_col='p',
        category_orders={
            'm': [1.0],
            'p': [100, 300, 500]
        },
        points='all'
    )

    fig_time.update_layout(
        title='Execution time vs Density',
        yaxis_title='Execution time (seconds)',
        boxmode='group'
    )
    fig_time.write_html(f"roots_screening_vs_direct_lingam_execution_time_{n}_samples.html")

## Experiment 3

In [41]:
df = pd.read_csv("./execution_time_comparison/exp3/root_screening_lingam_ba_network_exp3.csv")

In [44]:
df.head()

,precision_roots_screening,recall_roots_screening,mse_roots_screening,shd_roots_screening,precision_direct_lingam,recall_direct_lingam,mse_direct_lingam,shd_direct_lingam,p,m,n,random_state,exec_time_sec_roots_screening,exec_time_sec_direct_lingam
0,0.970588,1.0,0.000006,3,0.933962,1.0,0.000008,7,100,1,2000,0,213.406159,352.753299
1,1.000000,1.0,0.000004,0,0.970588,1.0,0.000005,3,100,1,2000,1,249.135848,403.082272
2,0.942857,1.0,0.000008,6,0.925234,1.0,0.000009,8,100,1,2000,2,200.116342,352.241102
3,0.990000,1.0,0.000004,1,0.908257,1.0,0.000009,10,100,1,2000,3,216.681923,364.140140
4,0.970588,1.0,0.000005,3,0.961165,1.0,0.000004,4,100,1,2000,4,208.766385,358.897836


In [45]:
# evaluate_experiments.py
# ------------------------------------------------------------
# Creates Plotly Express boxplots comparing:
#   - Roots Screening vs DirectLiNGAM
# for metrics:
#   - precision, recall, mse, shd, execution time
#
# Saves each figure as interactive HTML into an output folder.
# ------------------------------------------------------------

from __future__ import annotations

from pathlib import Path
import pandas as pd
import plotly.express as px


# ---------- config ----------
INPUT_CSV = "./execution_time_comparison/exp3/root_screening_lingam_ba_network_exp3.csv"  # change if needed
OUTPUT_DIR = "./execution_time_comparison/exp3/charts"  # change if needed

# Expected columns (as in your dataset)
MODEL_COLS = {
    "Roots Screening": {
        "precision": "precision_roots_screening",
        "recall": "recall_roots_screening",
        "mse": "mse_roots_screening",
        "shd": "shd_roots_screening",
        "exec_time_sec": "exec_time_sec_roots_screening",
    },
    "DirectLiNGAM": {
        "precision": "precision_direct_lingam",
        "recall": "recall_direct_lingam",
        "mse": "mse_direct_lingam",
        "shd": "shd_direct_lingam",
        "exec_time_sec": "exec_time_sec_direct_lingam",
    },
}

VARS = ["n", "p", "m"]


# ---------- helpers ----------
def ensure_dir(path: str | Path) -> Path:
    out = Path(path)
    out.mkdir(parents=True, exist_ok=True)
    return out


def to_long(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    Convert wide df into long format for a single metric with columns:
    [n, p, m, random_state, model, value]
    """
    rows = []
    for model_name, mapping in MODEL_COLS.items():
        col = mapping[metric]
        if col not in df.columns:
            raise KeyError(f"Missing column for metric '{metric}': {col}")
        tmp = df[["n", "p", "m", "random_state"]].copy()
        tmp["model"] = model_name
        tmp["value"] = df[col]
        rows.append(tmp)
    long_df = pd.concat(rows, ignore_index=True)
    return long_df


def choose_facets(df_long: pd.DataFrame, vary: str, max_levels: int = 8):
    """
    Pick facet_row and facet_col from the remaining variables (besides 'vary'),
    but only if the number of unique levels is not too large.
    """
    others = [v for v in VARS if v != vary]
    # Prefer faceting by p (columns) then m/n (rows) for readability
    # but keep it dynamic based on uniqueness.
    col_candidate, row_candidate = others[0], others[1]

    facet_col = col_candidate if df_long[col_candidate].nunique() <= max_levels else None
    facet_row = row_candidate if df_long[row_candidate].nunique() <= max_levels else None

    # If both are huge, drop both; if one is huge, keep the other.
    return facet_row, facet_col, others


def save_boxplot(
    df_long: pd.DataFrame,
    metric: str,
    vary: str,
    outdir: Path,
    title: str,
    fixed_filters: dict | None = None,
):
    """
    Save a Plotly Express boxplot (interactive HTML) comparing the two models.
    """
    d = df_long.copy()
    if fixed_filters:
        for k, v in fixed_filters.items():
            d = d.loc[d[k] == v].copy()

    facet_row, facet_col, others = choose_facets(d, vary=vary, max_levels=10)

    fig = px.box(
        d,
        x=vary,
        y="value",
        color="model",
        points="outliers",
        facet_row=facet_row,
        facet_col=facet_col,
        title=title,
        labels={"value": metric, "model": "Model"},
    )

    # slightly nicer layout for lots of facets
    fig.update_layout(boxmode="group", legend_title_text="Model")
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

    facet_tag = "none"
    if facet_row and facet_col:
        facet_tag = f"{facet_row}x{facet_col}"
    elif facet_row:
        facet_tag = f"{facet_row}"
    elif facet_col:
        facet_tag = f"{facet_col}"

    fname = f"metric_{metric}__vary_{vary}__facet_{facet_tag}.html"
    fig.write_html(str(outdir / fname), include_plotlyjs="cdn")
    print(f"[saved] {outdir / fname}")


def main():
    outdir = ensure_dir(OUTPUT_DIR)

    df = pd.read_csv(INPUT_CSV)

    # basic checks
    required = {"n", "p", "m", "random_state"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    # ---------- 1) Precision & Recall ----------
    for metric in ["precision", "recall"]:
        dlong = to_long(df, metric)
        for vary in ["n", "p", "m"]:
            save_boxplot(
                dlong,
                metric=metric,
                vary=vary,
                outdir=outdir,
                title=f"{metric.upper()} vs {vary} (Roots Screening vs DirectLiNGAM)",
            )

    # ---------- 2) MSE ----------
    dlong = to_long(df, "mse")
    for vary in ["n", "p", "m"]:
        save_boxplot(
            dlong,
            metric="mse",
            vary=vary,
            outdir=outdir,
            title=f"MSE vs {vary} (Roots Screening vs DirectLiNGAM)",
        )

    # ---------- 3) SHD ----------
    dlong = to_long(df, "shd")
    for vary in ["n", "p", "m"]:
        save_boxplot(
            dlong,
            metric="shd",
            vary=vary,
            outdir=outdir,
            title=f"SHD vs {vary} (Roots Screening vs DirectLiNGAM)",
        )

    # ---------- 4) Execution time (vary p and m only; keep n fixed) ----------
    # If multiple n values exist, we fix to the most frequent n (mode).
    n_fixed = df["n"].mode().iloc[0]
    dlong = to_long(df, "exec_time_sec")

    # vary p (facet by m if reasonable)
    save_boxplot(
        dlong,
        metric="exec_time_sec",
        vary="p",
        outdir=outdir,
        title=f"Execution Time vs p (n fixed at {n_fixed})",
        fixed_filters={"n": n_fixed},
    )

    # vary m (facet by p if reasonable)
    save_boxplot(
        dlong,
        metric="exec_time_sec",
        vary="m",
        outdir=outdir,
        title=f"Execution Time vs m (n fixed at {n_fixed})",
        fixed_filters={"n": n_fixed},
    )

    print("\nDone.")


if __name__ == "__main__":
    main()


[saved] execution_time_comparison/exp3/charts/metric_precision__vary_n__facet_mxp.html
[saved] execution_time_comparison/exp3/charts/metric_precision__vary_p__facet_mxn.html
[saved] execution_time_comparison/exp3/charts/metric_precision__vary_m__facet_pxn.html
[saved] execution_time_comparison/exp3/charts/metric_recall__vary_n__facet_mxp.html
[saved] execution_time_comparison/exp3/charts/metric_recall__vary_p__facet_mxn.html
[saved] execution_time_comparison/exp3/charts/metric_recall__vary_m__facet_pxn.html
[saved] execution_time_comparison/exp3/charts/metric_mse__vary_n__facet_mxp.html
[saved] execution_time_comparison/exp3/charts/metric_mse__vary_p__facet_mxn.html
[saved] execution_time_comparison/exp3/charts/metric_mse__vary_m__facet_pxn.html
[saved] execution_time_comparison/exp3/charts/metric_shd__vary_n__facet_mxp.html
[saved] execution_time_comparison/exp3/charts/metric_shd__vary_p__facet_mxn.html
[saved] execution_time_comparison/exp3/charts/metric_shd__vary_m__facet_pxn.html
[